# RockyBot rebuilt — Steps 1 & 2: Load and Split

Modern LangChain 1.x + Gemini. Everything free.

**Before running:** put your Google key in `rebuild/.env`:
```
GOOGLE_API_KEY=your_key_here
```


## Step 0 — Environment check

In [ ]:
import sys
print("python:", sys.executable)

import langchain, langchain_core, faiss, bs4, requests
print("langchain     ", langchain.__version__)
print("langchain-core", langchain_core.__version__)
print("faiss         ", faiss.__version__)
print("bs4           ", bs4.__version__)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
key = os.environ.get("GOOGLE_API_KEY", "")
assert key and key != "paste_your_key_here", "Put your key in rebuild/.env, then re-run"
print("key loaded:", key[:6] + "..." + key[-4:])

## Step 1 — Load the articles

The tutorial used `UnstructuredURLLoader`, which needs hundreds of MB of
dependencies. The obvious substitute, `WebBaseLoader`, lives in
`langchain-community` — **which is being sunset**, so we avoid it too.

Instead: ~10 lines using `requests` + BeautifulSoup. No extra dependency, no
deprecation, and you can see exactly what "scraping" means. `Document` comes
from `langchain-core`, which is the actively maintained foundation.

In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; learning-project)"}

def load_url(url: str) -> Document:
    """Fetch a page and return its readable text as a LangChain Document."""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # Strip the furniture: scripts, styling, menus, footers.
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    main = soup.find("main") or soup.find("article") or soup.body
    text = main.get_text(separator="\n", strip=True)
    title = soup.title.get_text(strip=True) if soup.title else url

    # metadata["source"] is what lets us cite the article later.
    return Document(page_content=text, metadata={"source": url, "title": title})


URLS = [
    "https://en.wikipedia.org/wiki/Tata_Motors",
    "https://en.wikipedia.org/wiki/Mahindra_%26_Mahindra",
]

docs = [load_url(u) for u in URLS]
print(f"loaded {len(docs)} documents\n")
for d in docs:
    print(f"  {d.metadata['title'][:45]:48s} {len(d.page_content):>7,} chars")

**Look at the raw text.** Scraped content is never pristine — that's normal.

In [ ]:
print(docs[0].page_content[:500])
print("\n--- metadata ---")
print(docs[0].metadata)

## Step 2 — Split into chunks

Why split?

1. A whole article won't fit in one request
2. **Retrieval works better on focused passages.** A 25,000-character document
   matches *every* question weakly. A 1,000-character chunk matches *one*
   question strongly.

`RecursiveCharacterTextSplitter` tries separators in order — paragraph breaks
first, then newlines, then sentences, then words. So it cuts at natural
boundaries instead of mid-word.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks\n")

sizes = [len(c.page_content) for c in chunks]
print(f"size  min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)//len(sizes)}")
print(f"chunks under 50 chars: {sum(1 for s in sizes if s < 50)}")

Note the sizes **vary** — they are not all 1000.

`chunk_size` is a *ceiling*, not a target. The splitter fills a chunk up to that
limit but stops at the last clean separator, so a chunk ending at a paragraph
break might only be 400 characters. Very short chunks are usually headings or
stray lines.

In [ ]:
c = chunks[107]
print("SOURCE:", c.metadata["source"])
print("LENGTH:", len(c.page_content))
print("-" * 60)
print(c.page_content)

### How much overlap do you actually get?

`chunk_overlap=200` asks consecutive chunks to share up to 200 characters, so a
sentence spanning a boundary is still findable.

But it's a **maximum, not a guarantee.** Overlap is carried over only when the
splitter merges several small pieces. When a chunk ends exactly on a paragraph
break, there may be almost none. Measure it rather than assume it:

In [ ]:
import difflib

print("chunk pair   size   shared chars")
for i in range(8):
    a, b = chunks[i].page_content, chunks[i + 1].page_content
    m = difflib.SequenceMatcher(None, a[-300:], b[:300]).find_longest_match(
        0, min(300, len(a)), 0, min(300, len(b))
    )
    print(f"  {i:2d} -> {i+1:<3d} {len(a):5d}   {m.size:3d}")

print("\nSome pairs overlap heavily, others barely at all — that is expected.")

---

## Checkpoint

`chunks` is now a list of passages, each remembering its source URL.

**Next:** turn each chunk into a vector (an embedding) and load them into a
FAISS index, so we can search by meaning rather than keyword.